## 1. EDA — первинне дослідження даних Olist

Мета цього ноутбука — дослідити 9 CSV-файлів датасету **до будь-якого очищення**: зрозуміти структуру кожної таблиці, виявити проблеми з якістю даних і сформулювати рішення для ETL-етапу.

Перевіряємо чотири категорії:
- **Типи даних** — чи правильно pandas розпізнав дати, числа, рядки
- **Пропуски (NULL)** — де і скільки, чи це норма чи аномалія
- **Дублікати та унікальність** — перевірка первинних ключів, повні дублікати рядків
- **Цілісність зв'язків (FK)** — чи всі значення дочірніх таблиць мають відповідники в батьківських

Результат — зведена таблиця у секції 1.6: кожна знайдена проблема, її критичність і рішення, що буде реалізоване в `02_etl.ipynb`.

---

### 1.1 Завантаження даних

Читаємо всі 9 CSV-файлів у словник DataFrames. На цьому кроці — тільки завантаження без будь-яких трансформацій. Виводимо розмір кожного файлу для звірки з описом датасету.

In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path.cwd().parent / "data" / "raw"

files = {
    'customers': 'olist_customers_dataset.csv',
    'orders': 'olist_orders_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'order_payments': 'olist_order_payments_dataset.csv',
    'order_reviews': 'olist_order_reviews_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv',
    'category_translation': 'product_category_name_translation.csv',
}

dfs = {}
for key, path in files.items():
    dfs[key] = pd.read_csv(DATA_DIR / path)

# Швидка перевірка — скільки рядків у кожному файлі
for name, df in dfs.items():
    print(f"{name:25s} → {df.shape[0]:>7,} rows, {df.shape[1]:>2} cols")

customers                 →  99,441 rows,  5 cols
orders                    →  99,441 rows,  8 cols
order_items               → 112,650 rows,  7 cols
order_payments            → 103,886 rows,  5 cols
order_reviews             →  99,224 rows,  7 cols
products                  →  32,951 rows,  9 cols
sellers                   →   3,095 rows,  4 cols
geolocation               → 1,000,163 rows,  5 cols
category_translation      →      71 rows,  2 cols


### 1.2 Детальний огляд кожного файлу

Для кожної таблиці виводимо стандартний набір діагностики: розмір (`shape`), типи колонок (`dtypes`), перші 5 рядків (`head`), кількість пропусків по колонках (`isnull().sum()`), кількість повних дублікатів рядків (`duplicated().sum()`), кількість унікальних значень по кожній колонці (`nunique()`) і базову статистику (`describe(include='all')`).

Звертаємо увагу на: тип `object` у колонках з датами, несподівані мінімуми/максимуми в числових полях, різницю між кількістю рядків і кількістю унікальних значень у потенційних первинних ключах.

In [2]:
def eda_summary(df, name):
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    print(f"\nShape: {df.shape}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nFirst 5 rows:")
    display(df.head())
    print(f"\nNull values:\n{df.isnull().sum()}")
    print(f"\nDuplicate rows: {df.duplicated().sum()}")
    print(f"\nUnique values:\n{df.nunique()}")
    print(f"\nDescribe:")
    display(df.describe(include='all'))

for name, df in dfs.items():
    eda_summary(df, name)


  customers

Shape: (99441, 5)

Dtypes:
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

First 5 rows:


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP



Null values:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Duplicate rows: 0

Unique values:
customer_id                 99441
customer_unique_id          96096
customer_zip_code_prefix    14994
customer_city                4119
customer_state                 27
dtype: int64

Describe:


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
count,99441,99441,99441.000000,99441,99441
unique,99441,96096,NaN,4119,27
top,06b8999e2fba1a1fbc88172c00ba8bc7,8d50f5eadf50201ccdcedfb9e2ac8455,NaN,sao paulo,SP
freq,1,17,NaN,15540,41746
mean,NaN,NaN,35137.474583,NaN,NaN
std,NaN,NaN,29797.938996,NaN,NaN
min,NaN,NaN,1003.000000,NaN,NaN
25%,NaN,NaN,11347.000000,NaN,NaN
50%,NaN,NaN,24416.000000,NaN,NaN
75%,NaN,NaN,58900.000000,NaN,NaN



  orders

Shape: (99441, 8)

Dtypes:
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

First 5 rows:


...

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
count,1.000163e+06,1.000163e+06,1.000163e+06,1000163,1000163
unique,NaN,NaN,NaN,8011,27
top,NaN,NaN,NaN,sao paulo,SP
freq,NaN,NaN,NaN,135800,404268
mean,3.657417e+04,-2.117615e+01,-4.639054e+01,NaN,NaN
std,3.054934e+04,5.715866e+00,4.269748e+00,NaN,NaN
min,1.001000e+03,-3.660537e+01,-1.014668e+02,NaN,NaN
25%,1.107500e+04,-2.360355e+01,-4.857317e+01,NaN,NaN
50%,2.653000e+04,-2.291938e+01,-4.663788e+01,NaN,NaN
75%,6.350400e+04,-1.997962e+01,-4.376771e+01,NaN,NaN



  category_translation

Shape: (71, 2)

Dtypes:
product_category_name            object
product_category_name_english    object
dtype: object

First 5 rows:


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor



Null values:
product_category_name            0
product_category_name_english    0
dtype: int64

Duplicate rows: 0

Unique values:
product_category_name            71
product_category_name_english    71
dtype: int64

Describe:


,product_category_name,product_category_name_english
count,71,71
unique,71,71
top,beleza_saude,health_beauty
freq,1,1


### 1.3 Перевірка первинних ключів

Перевіряємо унікальність первинних ключів у кожній таблиці. Для `order_items` — складений PK `(order_id, order_item_id)`, тому перевіряємо унікальність комбінації. Для `order_payments` — множинні платежі на одне замовлення є нормою, тому виводимо статистику без перевірки на унікальність `order_id`.

In [3]:
pk_checks = {
    'customers':            'customer_id',
    'orders':               'order_id',
    'products':             'product_id',
    'sellers':              'seller_id',
    'order_reviews':        'review_id',
    'category_translation': 'product_category_name',
}

for table, pk in pk_checks.items():
    is_unique = dfs[table][pk].is_unique
    status = '' if is_unique else ''
    print(f"{status} {table}.{pk} — unique: {is_unique}")

 customers.customer_id — unique: True
 orders.order_id — unique: True
 products.product_id — unique: True
 sellers.seller_id — unique: True
 order_reviews.review_id — unique: False
 category_translation.product_category_name — unique: True


In [4]:
# order_items — складений ключ (order_id + order_item_id)
dupes = dfs['order_items'].duplicated(subset=['order_id', 'order_item_id']).sum()
print(f"{'' if dupes == 0 else ''} order_items (order_id + order_item_id) — duplicates: {dupes}")

# order_payments — може мати кілька платежів на замовлення (це нормально)
print(f" order_payments — {dfs['order_payments']['order_id'].nunique()} unique orders, "
      f"{len(dfs['order_payments'])} rows (multiple payments per order expected)")

 order_items (order_id + order_item_id) — duplicates: 0
 order_payments — 99440 unique orders, 103886 rows (multiple payments per order expected)


### 1.4 Попередні спостереження

На основі кроків 1.2 та 1.3 зафіксовано наступні спостереження, які потребують поглибленої перевірки:

**Типи даних:**
- У таблицях `orders`, `order_items`, `order_reviews` усі колонки з датами мають тип `object` замість `datetime`. Це заважатиме розрахункам (час доставки, динаміка по місяцях). Потребує конвертації на етапі ETL.

**Пропуски (NULL):**
- `orders`: колонки `order_approved_at` (160), `delivered_carrier_date` (1783), `delivered_customer_date` (2965) мають пропуски. Потрібно перевірити — чи це лише незавершені замовлення (статус ≠ delivered), чи є аномалії серед доставлених.
- `order_reviews`: ~88% пропусків у `comment_title`, ~59% у `comment_message`. На перший погляд це нормально (не всі пишуть текст), але потрібно підтвердити.
- `products`: 610 пропусків одночасно в `category_name`, `name_lenght`, `description_lenght`, `photos_qty` — схоже, це ті самі 610 рядків з неповним заповненням. Окремо 2 пропуски у фізичних розмірах (weight, length, height, width).

**Первинні ключі:**
- `order_reviews.review_id` — NOT unique (98410 унікальних з 99224 рядків). Це критична проблема, бо за планом `review_id` є PK. Потрібно зрозуміти: це повні дублікати рядків чи різні відгуки з випадково однаковим ID?

**Підозрілі значення в describe():**
- `order_items`: `price` min=0.85, max=6735 при mean=120 — потенційні outliers у верхній частині. `freight_value` min=0.00 — є товари з безкоштовною доставкою, потрібно перевірити масштаб.
- `order_payments`: `payment_installments` min=0 — розстрочка на 0 платежів нелогічна. `payment_value` min=0.00 — нульові платежі потребують пояснення. Max 29 послідовних платежів на одне замовлення — можливий edge case.
- `products`: `product_weight_g` min=0 — товар з нульовою вагою нелогічний для фізичної доставки.

**Дублікати:**
- `geolocation`: 261831 повний дублікат (~26%). Також у describe() видно координати за межами Бразилії (lat max=+45, lng max=+121) — потрібно перевірити масштаб проблеми.

**Розбіжність довідників:**
- У `products` є 73 унікальні категорії, а в `category_translation` лише 71 переклад. 2 категорії залишаться без англійської назви після merge.

**Зв'язки між таблицями:**
- `customers` має 99441 `customer_id`, але лише 96096 `customer_unique_id` — один реальний клієнт може мати кілька технічних ID. Це впливає на розрахунок repeat rate (Питання 6 в аналізі).
- FK-цілісність поки не перевірена — потрібно переконатися, що всі `product_id`, `seller_id`, `order_id` в дочірніх таблицях мають відповідники в батьківських.

Далі проводимо поглиблену перевірку кожного з цих спостережень.

### 1.5 Поглиблена перевірка знайдених проблем

Перевіряємо кожне підозріле спостереження зі секції 1.4: NULL у датах доставлених замовлень, нульові значення в числових полях, дублікати у відгуках, нестандартні координати в geolocation та FK-цілісність між таблицями. Для кожної проблеми визначаємо масштаб і підтверджуємо рішення.

In [5]:
# orders — NULL у датах доставки для статусу 'delivered'
print(f"\nDelivered без approved_at: "
      f"{dfs['orders'][(dfs['orders']['order_status']=='delivered') & (dfs['orders']['order_approved_at'].isnull())].shape[0]}")
print(f"Delivered без delivered_customer_date: "
      f"{dfs['orders'][(dfs['orders']['order_status']=='delivered') & (dfs['orders']['order_delivered_customer_date'].isnull())].shape[0]}")

# orders — розподіл статусів (скільки не delivered)
print(dfs['orders']['order_status'].value_counts())


Delivered без approved_at: 14
Delivered без delivered_customer_date: 8
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [6]:
# order_items — нульовий freight і нульова/мінімальна ціна
print(f"price <= 0: {(dfs['order_items']['price'] <= 0).sum()}")
print(f"freight_value == 0: {(dfs['order_items']['freight_value'] == 0).sum()}")

price <= 0: 0
freight_value == 0: 383


In [7]:
# order_payments — нульові installments і нульовий value
print(f"installments == 0: {(dfs['order_payments']['payment_installments'] == 0).sum()}")
print(f"value == 0: {(dfs['order_payments']['payment_value'] == 0).sum()}")

# Які типи платежів мають 0 installments?
print(f"\nТипи платежів з 0 installments:")
print(dfs['order_payments'][dfs['order_payments']['payment_installments'] == 0]['payment_type'].value_counts())

installments == 0: 2
value == 0: 9

Типи платежів з 0 installments:
payment_type
credit_card    2
Name: count, dtype: int64


In [8]:
# order_reviews — дублікати review_id
review_dupes = dfs['order_reviews'][dfs['order_reviews'].duplicated(subset='review_id', keep=False)]
print(f"Рядків з дубльованим review_id: {len(review_dupes)}")
display(review_dupes.sort_values('review_id').head(10))

Рядків з дубльованим review_id: 1603


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44


In [9]:
# products — нульова вага
print(f"weight == 0: {(dfs['products']['product_weight_g'] == 0).sum()}")

# products — категорії без перекладу
cats_in_products = set(dfs['products']['product_category_name'].dropna().unique())
cats_in_translation = set(dfs['category_translation']['product_category_name'].unique())
print(f"\nКатегорії без перекладу: {cats_in_products - cats_in_translation}")

weight == 0: 4

Категорії без перекладу: {'portateis_cozinha_e_preparadores_de_alimentos', 'pc_gamer'}


In [10]:
# geolocation — координати за межами Бразилії
geo = dfs['geolocation']
outside = geo[(geo['geolocation_lat'] < -34) | (geo['geolocation_lat'] > 6) |
              (geo['geolocation_lng'] < -74) | (geo['geolocation_lng'] > -34)]
print(f"Точок за межами Бразилії: {len(outside)} ({len(outside)/len(geo)*100:.3f}%)")

Точок за межами Бразилії: 42 (0.004%)


In [11]:
# FK-валідація — чи всі order_items.product_id існують у products
orphan_products = set(dfs['order_items']['product_id']) - set(dfs['products']['product_id'])
print(f"order_items з неіснуючим product_id: {len(orphan_products)}")

orphan_sellers = set(dfs['order_items']['seller_id']) - set(dfs['sellers']['seller_id'])
print(f"order_items з неіснуючим seller_id: {len(orphan_sellers)}")

orphan_orders = set(dfs['order_items']['order_id']) - set(dfs['orders']['order_id'])
print(f"order_items з неіснуючим order_id: {len(orphan_orders)}")

orphan_reviews = set(dfs['order_reviews']['order_id']) - set(dfs['orders']['order_id'])
print(f"order_reviews з неіснуючим order_id: {len(orphan_reviews)}")

order_items з неіснуючим product_id: 0
order_items з неіснуючим seller_id: 0
order_items з неіснуючим order_id: 0
order_reviews з неіснуючим order_id: 0


### 1.6 Зведення знайдених проблем

| # | Файл | Проблема | Критичність | Рішення (Крок 3) | Деталі |
|---|------|----------|-------------|-------------------|--------|
| 1 | orders | Усі дати як `object` | Висока | `pd.to_datetime()` | 5 колонок з датами |
| 2 | orders | NULL у датах доставлених замовлень | Низька | Залишити — масштаб не впливає на аналіз | 14 delivered без approved_at, 8 без delivered_customer_date з 96478 |
| 3 | orders | ~3% замовлень не delivered | Низька | Зафіксувати; для аналізу доставки фільтрувати по status=delivered | shipped: 1107, canceled: 625, unavailable: 609, invoiced: 314, processing: 301, created: 5, approved: 2 |
| 4 | order_items | Дата `shipping_limit_date` як `object` | Висока | `pd.to_datetime()` | — |
| 5 | order_items | `freight_value` = 0 у 383 записах | Низька | Не потребує дій — безкоштовна доставка або самовивіз | price <= 0 відсутні (добре) |
| 6 | order_payments | `payment_installments` = 0 | Низька | Залишити або видалити 2 записи | Обидва — credit_card, мізерний масштаб |
| 7 | order_payments | `payment_value` = 0 | Низька | Залишити або видалити 9 записів | Технічні записи |
| 8 | order_payments | До 29 платежів на одне замовлення | Низька | Залишити — edge case | order_id `fa65dad...` має freq=29 |
| 9 | order_reviews | `review_id` не унікальний | Середня | Складений PK: `(review_id, order_id)` | 1603 рядки; один відгук прив'язаний до різних order_id того самого клієнта — не помилка, а особливість даних |
| 10 | order_reviews | NULL у коментарях | Низька | Залишити NULL | comment_title: 87656 (~88%), comment_message: 58247 (~59%) — не всі пишуть текст |
| 11 | order_reviews | Дати як `object` | Висока | `pd.to_datetime()` | review_creation_date, review_answer_timestamp |
| 12 | products | NULL у category + метаданих | Середня | Категорію → 'unknown', решта — залишити NULL | 610 рядків без category_name, name_lenght, description_lenght, photos_qty |
| 13 | products | NULL у фізичних розмірах | Низька | Заповнити медіаною або видалити | weight_g, length_cm, height_cm, width_cm: по 2 NULL |
| 14 | products | `product_weight_g` = 0 | Низька | Заповнити медіаною по категорії | 4 товари з нульовою вагою |
| 15 | products | 2 категорії без перекладу | Середня | Додати вручну | `pc_gamer` → 'pc_gamer', `portateis_cozinha_e_preparadores_de_alimentos` → 'portable_kitchen_food_processors' |
| 16 | geolocation | 261831 повних дублікатів | Середня | `drop_duplicates()` | ~26% від 1M рядків |
| 17 | geolocation | Координати за межами Бразилії | Низька | Фільтр: lat ∈ [-34, 6], lng ∈ [-74, -34] | Лише 42 точки (0.004%) — мінімальний масштаб |
| 18 | customers | `customer_unique_id` не унікальний | Низька | Для repeat rate використовувати `customer_unique_id` | 96096 унікальних vs 99441 customer_id |
| 19 | FK-зв'язки | Цілісність зв'язків | — | Не потребує дій | 0 orphans по всіх FK (order_items → products/sellers/orders, reviews → orders) |

---

### Висновок

В EDA було виявлено **19 спостережень** у 7 таблицях. Критичних проблем, які унеможливлюють аналіз, немає — датасет придатний до роботи після очищення.

**Ключові рішення для ETL (`02_etl.ipynb`):**
- Конвертувати дати з `object` → `datetime` у таблицях `orders`, `order_items`, `order_reviews` (5+2+2 колонки).
- Заповнити NULL у фізичних розмірах товарів медіаною по категорії; `product_category_name` NULL → `'unknown'`.
- Додати переклади для 2 категорій без відповідника в `category_translation`: `pc_gamer` і `portateis_cozinha_e_preparadores_de_alimentos`.
- Дедублікувати `order_reviews` по складеному ключу `(review_id, order_id)` — зберегти перший запис.
- Агрегувати `geolocation` медіаною координат по zip-коду: ~1M рядків → ~15k унікальних zip.
- Малозначні аномалії (freight=0, installments=0, value=0, 42 точки поза Бразилією) — залишити або видалити на рівні ETL відповідно до бізнес-логіки.

FK-цілісність між усіма таблицями підтверджена: 0 orphans. Дані готові до завантаження в PostgreSQL.